# CardioIA — Fase 4 — Ir Além 1## Ética e governança em visão computacional**Grupo Purkinje** — FIAP — Turma 2TIAOR-2026Miriã Leal Mantovani (RM567811) · João Pedro Santos Azevedo (RM566701)---Este notebook complementa o relatório `../docs/RELATORIO_IR_ALEM_1.md`.O notebook principal já reporta a distribuição das classes e o desempenho por classe no`classification_report`. O que falta lá não é o dado bruto — é a leitura de governança sobreele. Quatro medidas que só existem aqui:1. **Razões de disparidade** entre grupos, com a regra dos 80 % — transformar recall por   classe em um julgamento sobre equidade exige comparar as classes entre si, não apenas   listá-las.2. **Calibração (ECE)** — se a confiança declarada pelo modelo é honesta. Disso depende o   limiar de 60 % que aciona a revisão humana no protótipo.3. **Mitigação por limiar de custo** — quanto de precisão se troca por sensibilidade na   classe mais grave, medido em vez de estimado.4. **Intervalos de confiança de Wilson** — para que uma diferença entre grupos que é ruído   amostral não seja lida como desigualdade real.Antes disso, a seção 2 reúne as limitações do dataset: desbalanceamento, origem única eausência de metadados.### Por que fairness por classe diagnósticaA literatura de *fairness* normalmente agrupa por atributo protegido: sexo, faixa etária,etnia. **O dataset escolhido não traz nenhum desses campos.** Não há idade, não há sexo, nãohá identificador de paciente.Diante disso havia três saídas óbvias. Inventar atributos proxy a partir da imagem seria especulativoe produziria números bonitos sem significado. Abandonar a análise quantitativa custaria ocritério de avaliação. Trocar o dataset pelo NIH resolveria o eixo demográfico, mas ao custode abandonar o ECG — que é o exame cardiológico de verdade e o que dá sentido ao CardioIA.Escolhemos uma quarta: **medir disparidade entre as classes diagnósticas**. Em diagnósticoassistido, o grupo que sofre o dano não é definido só pela demografia — é definido tambémpela condição. Um modelo que acerta 98 % em exames normais e 70 % em infarto é injusto compacientes infartados, independentemente de quem eles sejam. As métricas são as mesmas(*equal opportunity*, *equalized odds*, razão de disparidade); o que muda é o eixo de grupo.**Registramos a limitação com honestidade:** esta análise não substitui auditoria demográfica.Ela mede um eixo real de injustiça, não todos. As seções 7 e 8 dizem o que ficou de fora e o queseria preciso coletar para fechar essa lacuna.

## 1. Carregamento do modelo e do conjunto de teste

In [ ]:
# Execute este notebook DEPOIS do CardioIA_Fase4_ECG_CNN — ele consome o modelo e o# conjunto de teste que aquele notebook produz.import os, json, itertoolsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport tensorflow as tffrom tensorflow import kerasfrom sklearn.metrics import confusion_matrix# Mesma detecção de ambiente do notebook principal: um arquivo só para Colab e local.try:    import google.colab          # noqa: F401    EM_COLAB = Trueexcept ImportError:    EM_COLAB = Falseif EM_COLAB:    BASE_DIR = '/content'else:    # O notebook mora em ir-alem-1/notebooks/; a raiz do repositório fica dois níveis acima.    _aqui = os.path.abspath('')    BASE_DIR = (os.path.abspath(os.path.join(_aqui, '..', '..'))                if os.path.basename(_aqui) == 'notebooks' else _aqui)OUT_DIR  = os.path.join(BASE_DIR, 'outputs')PROC_DIR = os.path.join(BASE_DIR, 'processed')FAIR_DIR = os.path.join(OUT_DIR, 'fairness')os.makedirs(FAIR_DIR, exist_ok=True)print('Ambiente:', 'Google Colab' if EM_COLAB else 'local (VS Code)')print('Base    :', BASE_DIR)def _ler_texto(caminho):    """Le um arquivo que pode ter sido gravado em UTF-8 ou em cp1252.    O notebook antigo gravava com a codificacao padrao do Windows; os arquivos    novos saem em UTF-8. Tentar os dois evita ter que reexecutar o treino so    por causa de um acento.    """    dados = open(caminho, 'rb').read()    for enc in ('utf-8', 'cp1252'):        try:            return dados.decode(enc)        except UnicodeDecodeError:            continue    return dados.decode('utf-8', errors='replace')# Mensagem útil em vez de FileNotFoundError cru: o motivo quase sempre é que o# notebook principal ainda não rodou até o fim (a célula que salva o modelo é a 2.3)._cfg_path = os.path.join(OUT_DIR, 'cardioia_ecg_config.json')if not os.path.exists(_cfg_path):    raise FileNotFoundError(        f'Não encontrei {_cfg_path}.\n'        'Este notebook depende dos artefatos do CardioIA_Fase4_ECG_CNN. Rode aquele '        'notebook até a seção 2.3 (que salva o modelo e o config.json) antes deste.\n'        'Se você rodou no Colab e está abrindo este aqui localmente, copie a pasta '        'outputs/ e a pasta processed/ para a raiz do repositório.'    )cfg = json.loads(_ler_texto(_cfg_path))CLASSES  = cfg['classes']DISPLAY  = cfg['display']IMG_SIZE = cfg['img_size']BATCH    = 32model = keras.models.load_model(os.path.join(OUT_DIR, 'cardioia_ecg_modelo.keras'))print('Modelo avaliado:', cfg['modelo'])test_ds = keras.utils.image_dataset_from_directory(    os.path.join(PROC_DIR, 'test'), labels='inferred', label_mode='int',    class_names=CLASSES, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=False)y_true = np.concatenate([y.numpy() for _, y in test_ds])probs  = model.predict(test_ds, verbose=0)y_pred = probs.argmax(1)print('Amostras de teste:', len(y_true))

## 2. Limitação 1 — o desbalanceamento é fabricado pela deduplicaçãoO primeiro viés de um dataset médico costuma estar na sua composição. Neste, ele não está àvista: **pelas contagens de arquivo o conjunto parece quase equilibrado** — 284 normais, 239de infarto agudo, 233 de batimento anormal, 172 de histórico de infarto, uma razão de 1,65× entre a maior e a menor.O desequilíbrio só aparece depois de remover as duplicatas exatas. O dataset repete imagensbyte a byte, e repete de forma muito desigual entre as classes: na de infarto agudo, cadaimagem aparece **oito vezes**. Dos 239 arquivos sobram 30 imagens distintas.Medimos as duas contagens lado a lado, porque a diferença entre elas é o achado.

In [ ]:
dist = {}for split in ['train', 'val', 'test']:    d = os.path.join(PROC_DIR, split)    dist[split] = {c: len(os.listdir(os.path.join(d, c))) for c in CLASSES}dist_df = pd.DataFrame(dist).rename(index=DISPLAY)dist_df['total'] = dist_df.sum(1)dist_df['% do total'] = (100 * dist_df.total / dist_df.total.sum()).round(1)display(dist_df)razao = dist_df.total.max() / dist_df.total.min()print(f'\nRazão de desbalanceamento: {razao:.2f}x '      f'({dist_df.total.idxmax()} ÷ {dist_df.total.idxmin()})')print('Referência usual: acima de 1,5x já pede compensação; acima de 4x o modelo tende a '      'ignorar a classe menor se nada for feito.')ax = dist_df[['train', 'val', 'test']].plot(kind='barh', stacked=True, figsize=(9, 3.6),                                            color=['#b3221e', '#e07a5f', '#f2cc8f'])ax.set_xlabel('nº de imagens'); ax.set_ylabel('')ax.set_title('Distribuição das classes por conjunto')plt.tight_layout(); plt.savefig(f'{FAIR_DIR}/f01_distribuicao_classes.png', dpi=150); plt.show()# Compara o que o dataset promete (arquivos) com o que entrega (imagens distintas).# Só roda se raw/ estiver disponível; o manifest basta para o resto do notebook.import hashlib, glob, collections_raw = os.path.join(BASE_DIR, 'raw')if os.path.isdir(_raw):    def _md5(p):        return hashlib.md5(open(p, 'rb').read()).hexdigest()    _por_pasta = collections.defaultdict(list)    for _p in glob.glob(os.path.join(_raw, '**', '*'), recursive=True):        if _p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):            _por_pasta[os.path.basename(os.path.dirname(_p))].append(_p)    _linhas = []    for _pasta, _arqs in sorted(_por_pasta.items()):        _uni = len({_md5(x) for x in _arqs})        _linhas.append({'pasta': _pasta[:34], 'arquivos': len(_arqs),                        'distintas': _uni, 'duplicatas': len(_arqs) - _uni,                        'fator': round(len(_arqs) / _uni, 1)})    _dup = pd.DataFrame(_linhas).set_index('pasta')    display(_dup)    _ra = _dup.arquivos.max() / _dup.arquivos.min()    _ru = _dup.distintas.max() / _dup.distintas.min()    print(f'\nRazão de desbalanceamento pelas CONTAGENS DE ARQUIVO : {_ra:.2f}x')    print(f'Razão de desbalanceamento pelas IMAGENS DISTINTAS    : {_ru:.2f}x')    print(f'\nO dataset aparenta ser {_ra:.1f}x desbalanceado e é {_ru:.1f}x.')    print('Um grupo que não deduplicasse não veria nem o desequilíbrio nem o vazamento.')else:    print('raw/ não encontrado — pulando a comparação com as contagens de arquivo.')    print('A razão acima já é a real, calculada sobre as imagens distintas.')

## 3. Métricas de fairness por grupo diagnósticoPara cada classe $c$ tratada como grupo, calculamos:| Métrica | Definição | O que denuncia ||---|---|---|| **TPR** (*equal opportunity*) | $P(\hat{y}=c \mid y=c)$ | O modelo deixa de reconhecer essa condição || **FPR** | $P(\hat{y}=c \mid y \neq c)$ | O modelo atribui essa condição a quem não a tem || **PPV** | $P(y=c \mid \hat{y}=c)$ | Quanto vale um laudo positivo para essa condição || **Taxa de seleção** | $P(\hat{y}=c)$ | *Demographic parity* — o quanto o modelo "gosta" da classe |A paridade perfeita (TPR igual para todas as classes) **não é o alvo clínico**. Uma seçãoadiante trata disso: num contexto em que errar infarto custa mais que errar normal, a métricaque importa é o piso de TPR nas classes graves, não a igualdade entre elas.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))N = cm.sum()linhas = []for i, c in enumerate(CLASSES):    TP = cm[i, i]    FN = cm[i].sum() - TP    FP = cm[:, i].sum() - TP    TN = N - TP - FN - FP    linhas.append({        'grupo': DISPLAY[c],        'suporte': int(cm[i].sum()),        'TPR': TP / (TP + FN) if TP + FN else np.nan,        'FPR': FP / (FP + TN) if FP + TN else np.nan,        'PPV': TP / (TP + FP) if TP + FP else np.nan,        'taxa_selecao': (TP + FP) / N,    })fair = pd.DataFrame(linhas).set_index('grupo')display(fair.round(3))fair.round(4).to_csv(f'{FAIR_DIR}/metricas_fairness.csv')

### 3.1 Razões de disparidadeDuas leituras consagradas, adaptadas ao eixo diagnóstico:- **Razão de igualdade de oportunidade** = menor TPR ÷ maior TPR. A *regra dos 80 %*, vinda do  direito antidiscriminatório norte-americano, trata razões abaixo de 0,80 como indício de  impacto desproporcional.- **Diferença de equalized odds** = maior diferença entre grupos, considerando TPR e FPR juntos.  Quanto mais perto de zero, mais uniforme é o comportamento do modelo.

In [ ]:
tpr_min, tpr_max = fair.TPR.min(), fair.TPR.max()razao_eo = tpr_min / tpr_maxdif_tpr  = tpr_max - tpr_mindif_fpr  = fair.FPR.max() - fair.FPR.min()eq_odds  = max(dif_tpr, dif_fpr)print(f'Razão de igualdade de oportunidade : {razao_eo:.3f}  '      f'({fair.TPR.idxmin()} vs {fair.TPR.idxmax()})')print(f'Diferença de TPR entre grupos      : {dif_tpr:.3f}')print(f'Diferença de FPR entre grupos      : {dif_fpr:.3f}')print(f'Diferença de equalized odds        : {eq_odds:.3f}')print()print('APROVADO na regra dos 80%.' if razao_eo >= 0.80 else      'REPROVADO na regra dos 80% — há disparidade relevante entre classes.')fig, ax = plt.subplots(figsize=(9, 4))x = np.arange(len(fair))ax.bar(x - 0.2, fair.TPR, 0.4, label='TPR (sensibilidade)', color='#b3221e')ax.bar(x + 0.2, fair.FPR, 0.4, label='FPR', color='#8d99ae')ax.axhline(0.80, ls='--', c='gray', lw=1)ax.text(len(fair) - 0.45, 0.815, 'referência 0,80', fontsize=8, color='gray')ax.set_xticks(x); ax.set_xticklabels(fair.index, rotation=18, ha='right')ax.set_ylim(0, 1.05); ax.set_ylabel('taxa'); ax.legend()ax.set_title('Desempenho por grupo diagnóstico')plt.tight_layout(); plt.savefig(f'{FAIR_DIR}/f02_tpr_fpr_por_grupo.png', dpi=150); plt.show()

## 4. O custo assimétrico do erroParidade estatística esconde a questão clínica. Um falso negativo de infarto manda para casaalguém que está infartando; um falso positivo gera um exame a mais. **Os dois contam igual nasmétricas acima e não valem a mesma coisa na vida do paciente.**Medimos, portanto, a matriz de erros em termos clínicos: para onde escorrega cada condiçãograve quando o modelo erra.

In [ ]:
GRAVES = ['IAM', 'Historico_IAM']idx_normal = CLASSES.index('Normal')print('Erros mais caros — condição grave classificada como Normal:\n')total_perigoso = 0for c in GRAVES:    i = CLASSES.index(c)    n = int(cm[i, idx_normal]); sup = int(cm[i].sum())    total_perigoso += n    print(f'  {DISPLAY[c]:<28} {n:>3} de {sup:>3} '          f'({100*n/sup:.1f}% dos casos) foram ditos Normal')print(f'\nTotal de falsos negativos clinicamente perigosos: {total_perigoso}')print(f'Como fração de todos os erros: {100*total_perigoso/(N-np.trace(cm)):.1f}%'      if N - np.trace(cm) else 'Nenhum erro cometido.')cm_norm = cm / cm.sum(1, keepdims=True)plt.figure(figsize=(6.8, 5.4))sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Reds',            xticklabels=[DISPLAY[c] for c in CLASSES],            yticklabels=[DISPLAY[c] for c in CLASSES])plt.xlabel('Previsto'); plt.ylabel('Real')plt.title('Matriz de confusão normalizada por linha')plt.xticks(rotation=30, ha='right'); plt.tight_layout()plt.savefig(f'{FAIR_DIR}/f03_matriz_normalizada.png', dpi=150); plt.show()

## 5. Calibração: a confiança do modelo é honesta?Um modelo mal calibrado é um problema de governança, não só de estatística: se ele diz "92 %"quando acerta 70 % das vezes, o limiar de revisão humana do protótipo (60 %) deixa passar casosque deveriam parar para revisão.Medimos o **Erro de Calibração Esperado (ECE)**: a média ponderada da distância entre confiançadeclarada e acerto observado, em faixas de confiança.

In [ ]:
conf = probs.max(1)acerto = (y_pred == y_true).astype(float)bins = np.linspace(0, 1, 11)idx = np.digitize(conf, bins) - 1ece, tabela = 0.0, []for b in range(10):    m = idx == b    if not m.sum():        continue    c_med, a_med = conf[m].mean(), acerto[m].mean()    ece += m.sum() / len(conf) * abs(c_med - a_med)    tabela.append({'faixa': f'{bins[b]:.1f}–{bins[b+1]:.1f}', 'n': int(m.sum()),                   'confianca_media': c_med, 'acuracia_observada': a_med})cal = pd.DataFrame(tabela)display(cal.round(3))print(f'\nECE = {ece:.4f}   (0 = perfeitamente calibrado; acima de 0,10 é preocupante)')plt.figure(figsize=(5.4, 5))plt.plot([0, 1], [0, 1], ls='--', c='gray', label='calibração perfeita')plt.plot(cal.confianca_media, cal.acuracia_observada, 'o-', c='#b3221e', label='modelo')plt.xlabel('confiança declarada'); plt.ylabel('acurácia observada')plt.title(f'Curva de confiabilidade — ECE {ece:.3f}')plt.legend(); plt.tight_layout()plt.savefig(f'{FAIR_DIR}/f04_calibracao.png', dpi=150); plt.show()# Quantos casos o limiar de revisão humana realmente capturaLIMIAR = 0.60abaixo = conf < LIMIARprint(f'\nCom limiar de {LIMIAR:.0%}: {abaixo.sum()} exames ({100*abaixo.mean():.1f}%) '      f'iriam para revisão humana.')if abaixo.sum():    print(f'Entre eles, a acurácia do modelo é {100*acerto[abaixo].mean():.1f}% — '          f'contra {100*acerto[~abaixo].mean():.1f}% acima do limiar.')

## 6. Mitigação: o que já foi feito e o que ainda falta### Já aplicado no pipeline principal| Prática | Onde | Efeito pretendido ||---|---|---|| Pesos por classe no treino | `CLASS_WEIGHT` | Impede que a classe menor seja ignorada pela função de perda || Divisão estratificada | `train_test_split(stratify=...)` | Mantém a proporção das classes nos três conjuntos || Remoção de duplicatas por hash | Parte 1.2 | Evita a mesma imagem em treino e teste, que inflaria a métrica || Aumento de dados sem espelhamento | `make_augment` | Espelhar um ECG cria um exame fisiologicamente impossível || Limiar de revisão humana | protótipo | Devolve ao humano os casos em que o modelo está em dúvida |### Experimento de mitigação: limiar por custoA última linha de defesa é ajustar o limiar de decisão da classe grave. Em vez de escolher omaior *softmax*, acionamos IAM sempre que sua probabilidade passar de um limiar menor —trocando precisão por sensibilidade de propósito.

In [ ]:
i_iam = CLASSES.index('IAM')linhas = []for limiar in [0.50, 0.40, 0.30, 0.25, 0.20, 0.15]:    pred = np.where(probs[:, i_iam] >= limiar, i_iam, probs.argmax(1))    TP = ((pred == i_iam) & (y_true == i_iam)).sum()    FN = ((pred != i_iam) & (y_true == i_iam)).sum()    FP = ((pred == i_iam) & (y_true != i_iam)).sum()    linhas.append({'limiar_IAM': limiar,                   'recall_IAM': TP / (TP + FN) if TP + FN else np.nan,                   'precisao_IAM': TP / (TP + FP) if TP + FP else np.nan,                   'falsos_negativos_IAM': int(FN),                   'acuracia_global': (pred == y_true).mean()})tr = pd.DataFrame(linhas)display(tr.round(3))tr.round(4).to_csv(f'{FAIR_DIR}/limiar_por_custo.csv', index=False)fig, ax = plt.subplots(figsize=(8, 4))ax.plot(tr.limiar_IAM, tr.recall_IAM, 'o-', label='recall de IAM', c='#b3221e')ax.plot(tr.limiar_IAM, tr.precisao_IAM, 's-', label='precisão de IAM', c='#8d99ae')ax.plot(tr.limiar_IAM, tr.acuracia_global, '^--', label='acurácia global', c='#3d405b')ax.invert_xaxis(); ax.set_xlabel('limiar aplicado à classe IAM'); ax.set_ylabel('taxa')ax.set_title('Trocar precisão por sensibilidade na classe mais grave')ax.legend(); plt.tight_layout()plt.savefig(f'{FAIR_DIR}/f05_limiar_por_custo.png', dpi=150); plt.show()

## 7. O que não conseguimos medirEsta é a seção mais importante do ponto de vista de governança, porque é a que impede que osnúmeros acima sejam lidos como um atestado de idoneidade do modelo.**Não há metadados demográficos.** Sem idade, sexo ou etnia, não é possível verificar se omodelo funciona pior para mulheres — pergunta que não é hipotética em cardiologia, onde aapresentação do infarto no ECG difere entre sexos e o subdiagnóstico em mulheres é um problemadocumentado. Nosso modelo pode reproduzir esse viés sem que nenhuma métrica deste notebookacuse.**A origem é única.** O dataset vem de uma única instituição, com um parque de equipamentos eum protocolo de aquisição. Um modelo treinado assim aprende também as características doaparelho. Em outro hospital, com outro eletrocardiógrafo, o desempenho pode cair sem aviso.**Não há identificador de paciente.** Se o mesmo paciente tiver mais de um exame no dataset,imagens correlacionadas podem ter caído em conjuntos diferentes. A remoção de duplicatas porhash pega o arquivo idêntico, não o segundo exame do mesmo paciente.**O conjunto de teste é pequeno.** Com poucas dezenas de imagens por classe, cada erro move amétrica em vários pontos percentuais. Os intervalos de confiança são largos, e diferençaspequenas entre grupos não devem ser lidas como reais.

In [ ]:
# Intervalo de confiança de Wilson: deixa explícita a incerteza de cada TPRdef wilson(k, n, z=1.96):    if n == 0:        return (np.nan, np.nan)    p = k / n    d = 1 + z**2 / n    c = (p + z**2 / (2*n)) / d    e = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d    return (max(0, c - e), min(1, c + e))print('TPR por grupo, com intervalo de confiança de 95% (Wilson):\n')for i, c in enumerate(CLASSES):    TP, sup = int(cm[i, i]), int(cm[i].sum())    lo, hi = wilson(TP, sup)    print(f'  {DISPLAY[c]:<28} {TP/sup:.3f}  [{lo:.3f}, {hi:.3f}]  (n={sup})')print('\nSe os intervalos de dois grupos se sobrepõem, a diferença de TPR entre eles')print('não é estatisticamente distinguível com este tamanho de amostra.')

## 8. Recomendações1. **Não usar em produção sem auditoria demográfica.** Antes de qualquer uso real, o dataset   precisa de idade e sexo, e as métricas da seção 3 precisam ser recalculadas nesses eixos.2. **Validação externa obrigatória.** Testar em exames de outra instituição, com outro   equipamento, antes de afirmar qualquer generalização.3. **Manter o humano na decisão.** O protótipo classifica e sinaliza; o laudo é do médico. O   limiar de revisão existe para isso e deve ser recalibrado com os números da seção 5.4. **Registrar a proveniência.** Instituição, período, equipamento e protocolo de aquisição   fazem parte da documentação do modelo, não do dataset apenas.5. **Monitorar após o deploy.** Desempenho por classe acompanhado ao longo do tempo detecta   desvio de distribuição — equipamento novo, protocolo alterado, perfil de paciente diferente.### Enquadramento legalDados de saúde são **dados pessoais sensíveis** pela LGPD (Lei 13.709/2018, Art. 5º, II), e seutratamento exige hipótese específica do Art. 11. O dataset usado aqui é público edesidentificado, o que afasta o tratamento de dado pessoal nesta etapa acadêmica — mas umsistema real que receba o ECG de um paciente identificado entra integralmente no escopo da lei,incluindo o direito à revisão de decisões automatizadas previsto no Art. 20.

In [ ]:
# Consolida os resultados num JSON, para o relatório citar números de execução realresumo = {    'n_teste': int(len(y_true)),    'razao_desbalanceamento': float(razao),    'tpr_por_grupo': {DISPLAY[c]: float(fair.TPR.iloc[i]) for i, c in enumerate(CLASSES)},    'fpr_por_grupo': {DISPLAY[c]: float(fair.FPR.iloc[i]) for i, c in enumerate(CLASSES)},    'razao_igualdade_oportunidade': float(razao_eo),    'diferenca_equalized_odds': float(eq_odds),    'aprovado_regra_80': bool(razao_eo >= 0.80),    'ece': float(ece),    'falsos_negativos_graves_como_normal': int(total_perigoso),}with open(f'{FAIR_DIR}/resumo_fairness.json', 'w', encoding='utf-8') as fh:    json.dump(resumo, fh, ensure_ascii=False, indent=2)print(json.dumps(resumo, ensure_ascii=False, indent=2))print(f'\nFiguras e tabelas salvas em {FAIR_DIR}')